# EX03: Decorators

In [72]:
import time

def log_action(f):
    def check_time(*args, **kwargs):
        print(f"Method '{f.__name__}' started on {(time.time())}.")
        res = f(*args, **kwargs)
        print(f"Method '{f.__name__}' finished on {(time.time())}.")
        return res
    return check_time

def log_time(f):
    def check_time(*args, **kwargs):
        start = time.time()
        res = f(*args, **kwargs)
        print(f"Method '{f.__name__}' executed for {time.time() - start}.")
        return res
    return check_time

# EX01: CSVReader

In [76]:
import pandas as pd
import numpy as np
import statistics

class CSVReader():
    def __init__(self, fp):
        self.fp = fp

    @log_action
    def read(self):
        return pd.read_csv(self.fp)
        
    @log_time
    def preview(self, n = 5):
        # preview only, so you're getting None of the data :^]
        # unless if you get the data from .read()
        # please don't do that, if you do so .preview() would be completely useless :^[
        print(self.read()[ : n])

In [77]:
test = CSVReader(r"./sample_data.csv")

In [75]:
test.read()

Method 'read' started on 1792981337.5484734.
Method 'read' finished on 1792981337.5484734.


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [78]:
test.preview()

Method 'read' started on 1792981361.8608913.
Method 'read' finished on 1792981361.8608913.
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
Method 'preview' executed for 0.014404773712158203.


# EX02: APfMVH

In [104]:
class MissingValueStrategy():
    def __init__(self):
        pass

    @log_action
    def handle(self, df):
        pass

In [105]:
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()

In [106]:
class FillMean(MissingValueStrategy):
    def handle(self, df):
        for col in df:
            ser = df[col]
            if True in ser.isna().values:
                temp = ser.dropna()
                df[col] = df[col].fillna(sum(temp)/len(temp))
        return df

In [107]:
class FillMode(MissingValueStrategy):
    def handle(self, df):
        for col in df:
            ser = df[col]
            if True in ser.isna().values:
                df[col] = df[col].fillna(statistics.mode(df[col]))
        return df

In [108]:
class DataCleaner():
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def do_mean(self, df):
        return self.strategy.handle(self, df)

In [109]:
d = DataCleaner(FillMean)

In [110]:
d.do_mean(test.read())

Method 'read' started on 1792982516.7793818.
Method 'read' finished on 1792982516.7808955.


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


In [32]:
d.strategy = FillMode
d.do_mean(test.read())

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,29.0,172.0,68.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,85.0
3,4,David,29.0,165.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,85.0
6,7,Grace,29.0,162.0,60.0,Chicago,92.0


# EX04: Factory Method

In [112]:
class DataTransform:
    @log_action
    def apply(self, df):
        pass

In [113]:
class NormalizeColumns(DataTransform):
    def apply(self, df):
        for col in df.columns:
            t = df[col]
            if t.dtype == 'float64':
                df[col] = (t - t.mean()) / t.std()
        return df

In [114]:
class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates()

In [115]:
class StandardizeText(DataTransform):
    def apply(self, df):
        return df # ?????????????????????????????????

In [119]:
class TransformFactory:
    def transform(self, df, method):
        method = self._get_method(method)
        return method.apply(df)

    def _get_method(self, method):
        if "norm" in method:
            return NormalizeColumns()
        elif "dup" in method:
            return RemoveDuplicates()
        elif "text" in method:
            return StandardizeText()

In [120]:
TransformFactory().transform(test.read(), "norm")

Method 'read' started on 1792983463.264744.
Method 'read' finished on 1792983463.3687263.


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,-0.367277,-0.345033,-0.174342,New York,-0.149906
1,2,Bob,NaN,0.862582,NaN,Los Angeles,0.599625
2,3,Charlie,1.395651,0.172516,0.174342,Chicago,NaN
3,4,David,NaN,NaN,0.871710,Houston,-1.648970
4,5,Eva,-0.954919,-1.207615,-1.307565,New York,0.299813
5,6,Frank,-0.073455,1.380131,1.307565,Los Angeles,NaN
6,7,Grace,NaN,-0.862582,-0.871710,Chicago,0.899438


# EX05: Template Method

In [138]:
class DataPipeline:
    def __init__(self, path):
        self.path = path
    
    def run(self, clean, transform):
        data = self.read(self.path)
        data = self.clean(data, clean)
        data = self.transform(data, transform)
        self.save(data)
        return data

    def read(self, path): ...
    def clean(self, data, clean): ...
    def transform(self, data, transform): ...
    def save(self, data):
        data.to_csv(self.path)

In [143]:
class CSVDataPipeline(DataPipeline):
    def read(self, path):
        return CSVReader(path).read()

    def clean(self, data, clean):
        method = self._get_clean(clean)
        return method.handle(data)

    def _get_clean(self, clean):
        if "drop" in clean:
            return DropMissing()
        elif "mean" in clean:
            return FillMean()
        elif "mode" in clean:
            return FillMode()

    def transform(self, data, transform):
        return TransformFactory().transform(data, transform)

In [145]:
CSVDataPipeline("./sample_data.csv").run("mean", "text")

Method 'read' started on 1792983817.9059615.
Method 'read' finished on 1792983818.005183.


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0
